# K-Nearest Neighbors (KNN) Classifier

**K-Nearest Neighbors (KNN)** is one of the simplest and most intuitive machine learning algorithms. It is a **non-parametric**, **instance-based (lazy) learning** algorithm that makes predictions based on the proximity of a new data point to existing training samples.

In these notes, we will cover:
1. **Core Intuition:** The "birds of a feather" principle and how KNN classifies data.
2. **Distance Metrics:** Euclidean, Manhattan, and Minkowski distances.
3. **The Critical Role of Feature Scaling.**
4. **Selecting the Optimal K:** Heuristics, cross-validation, and decision surface analysis.
5. **Overfitting vs. Underfitting:** How K controls model complexity.
6. **Limitations:** Latency, curse of dimensionality, outliers, imbalanced data, and lack of interpretability.
7. **Practical Python Demos:** Implementation using Scikit-Learn on real-world data.

## 1. KNN Intuition

### The Core Principle
KNN is based on the intuitive concept: *"You are the average of the five people you spend the most time with."* In data science terms, **similar data points exist close to each other in feature space**. If a new unknown data point is surrounded by mostly "Class A" points, it is likely also "Class A".

### How KNN Works (Step-by-Step)
1. **Choose a value for $K$** (the number of nearest neighbors to consider, e.g., $K = 5$).
2. **Calculate the distance** between the new query point and **every single** training data point.
3. **Sort** all distances in ascending order.
4. **Select the $K$ closest neighbors** (the $K$ training points with the smallest distances).
5. **Apply Majority Voting:** Count the class labels of the $K$ neighbors. The class with the highest count becomes the prediction.

### Key Characteristics
- **Non-Parametric:** KNN makes no assumptions about the underlying data distribution (unlike Naive Bayes which assumes Gaussian, or Logistic Regression which assumes linearity). It adapts entirely to the training data.
- **Lazy Learner:** KNN does **not have a training phase**. It simply memorizes all training data. All computation happens at **prediction time**, which makes it fast to train but slow to predict.
- **Instance-Based:** The model *is* the training data itself. There are no learned parameters (no weights, no coefficients).

## 2. Distance Metrics

The entire KNN algorithm revolves around measuring "closeness" between data points. Several distance metrics can be used:

### 1. Euclidean Distance (L2 Norm)
The straight-line distance between two points. This is the **default and most commonly used** metric.

$$d(P, Q) = \sqrt{\sum_{j=1}^{d} (p_j - q_j)^2}$$

### 2. Manhattan Distance (L1 Norm)
The sum of absolute differences along each dimension ("city block" distance).

$$d(P, Q) = \sum_{j=1}^{d} |p_j - q_j|$$

### 3. Minkowski Distance (Generalized)
A generalization that includes both Euclidean ($p=2$) and Manhattan ($p=1$) as special cases.

$$d(P, Q) = \left( \sum_{j=1}^{d} |p_j - q_j|^p \right)^{1/p}$$

| Metric | Formula Parameter | Behavior |
| :--- | :---: | :--- |
| Manhattan | $p = 1$ | Axis-aligned, robust to outliers in individual dimensions |
| Euclidean | $p = 2$ | Standard straight-line distance |
| Chebyshev | $p = \infty$ | Maximum difference along any single dimension |

## 3. The Critical Role of Feature Scaling

### Why Scaling is Mandatory for KNN
Since KNN is fundamentally a **distance-based** algorithm, the scale of each feature directly impacts the distance computation.

#### Example: Without Scaling
Suppose we have two features:
- **Income:** Ranges from $20{,}000$ to $200{,}000$.
- **Age:** Ranges from $18$ to $70$.

When computing Euclidean distance, the Income dimension will completely **dominate** the calculation because its numerical range is thousands of times larger than Age. The Age feature becomes virtually irrelevant.

### Solution: StandardScaler
Apply `StandardScaler` (or `MinMaxScaler`) to bring all features to a comparable scale **before** training:

$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$

This ensures each feature contributes proportionally to the distance calculation.

> **Important:** Always fit the scaler on the **training set only**, then transform both training and test sets using the same scaler to prevent data leakage.

## 4. Selecting the Optimal K

The value of $K$ is the single most important hyperparameter in KNN. Choosing it correctly is critical.

### Approach 1: Heuristic (Rule of Thumb)
A common starting point is:

$$K = \sqrt{n}$$

Where $n$ is the total number of training samples. For example, if $n = 100$, start with $K = 10$.

### Approach 2: Cross-Validation (Recommended)
Systematically evaluate model performance for a range of $K$ values (e.g., $K = 1$ to $K = 50$) using **k-fold cross-validation** and plot the accuracy (or error rate) as a function of $K$. Select the $K$ that gives the best validation performance.

### Approach 3: Decision Surface Analysis
For 2D feature spaces, visualize the **decision surface** (the regions in the feature space assigned to each class) at different values of $K$. This helps intuitively understand how $K$ affects the complexity of the decision boundary.

### Practical Tips
- **Always choose an odd $K$** for binary classification to avoid ties in majority voting.
- Small $K$ captures local patterns (risk of overfitting); large $K$ captures global patterns (risk of underfitting).

## 5. Overfitting vs. Underfitting

### Low $K$ (e.g., $K = 1$): Overfitting
- The model considers only the **single closest neighbor**.
- The decision boundary becomes **extremely jagged and irregular**, conforming to every noise point and outlier in the training data.
- **High training accuracy, low test accuracy** (poor generalization).
- The model has **high variance** and **low bias**.

### High $K$ (e.g., $K = n$): Underfitting
- The model considers **all data points**, effectively always predicting the majority class.
- The decision boundary becomes **overly smooth or even a single region**, ignoring all local data structures.
- **Low training accuracy, low test accuracy**.
- The model has **low variance** and **high bias**.

### The Bias-Variance Trade-off in KNN

| $K$ Value | Bias | Variance | Boundary Complexity | Risk |
| :---: | :---: | :---: | :--- | :--- |
| Small (e.g., $1$) | Low | High | Very complex, jagged | Overfitting |
| Optimal | Balanced | Balanced | Smooth yet adaptive | Best generalization |
| Large (e.g., $n$) | High | Low | Overly simple, flat | Underfitting |

## 6. Limitations of KNN

Despite its simplicity and elegance, KNN has several significant practical limitations:

### 1. High Prediction Latency (Lazy Learner)
KNN has **zero training time** but extremely **high prediction time**. For every new query, it must compute the distance to *every* training point and sort them. For a dataset with $n$ samples and $d$ features, prediction complexity is $O(n \cdot d)$ per query. This makes KNN impractical for large-scale real-time applications.

### 2. Curse of Dimensionality
In high-dimensional spaces (e.g., $d > 20$ features), all data points tend to become roughly **equidistant** from each other. The concept of "nearest neighbor" loses meaning because the ratio of distances between the closest and farthest points approaches $1$. The algorithm's performance degrades significantly as dimensionality increases.

### 3. Sensitivity to Outliers
With small values of $K$, a single outlier in the neighborhood can flip the predicted class. For example, if $K = 3$ and one of the three neighbors is a noisy/mislabeled point, it may cause a misclassification.

### 4. Feature Scaling Dependency
As discussed in Section 3, KNN is completely dependent on proper feature scaling. Without standardization, features with larger numerical ranges will disproportionately dominate the distance metric.

### 5. Sensitivity to Imbalanced Data
If one class has significantly more samples than another, the majority class will dominate the neighborhood of most query points, biasing predictions toward the majority class. **Weighted KNN** (giving closer neighbors higher voting weight) can partially mitigate this.

### 6. Lack of Interpretability (Black Box)
KNN provides no insight into *why* a prediction was made. There are no learned coefficients, feature importances, or decision rules. It acts as a "black box" — it gives an answer but not an explanation.

## 7. Code Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set visualization contexts
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 8. Demo A: KNN on the Breast Cancer Dataset

We will demonstrate KNN classification on Scikit-Learn's built-in **Breast Cancer Wisconsin** dataset (30 features, 2 classes: malignant vs. benign). This demo highlights the importance of feature scaling.

In [ ]:
# 1. Load dataset
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

# 2. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. WITHOUT scaling (to demonstrate the problem)
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_unscaled.fit(X_train, y_train)
y_pred_unscaled = knn_unscaled.predict(X_test)
acc_unscaled = accuracy_score(y_test, y_pred_unscaled)

# 4. WITH scaling (correct approach)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = knn_scaled.predict(X_test_scaled)
acc_scaled = accuracy_score(y_test, y_pred_scaled)

# 5. Compare
print("Impact of Feature Scaling on KNN (Breast Cancer Dataset)")
print("="*55)
print(f"Accuracy WITHOUT scaling: {acc_unscaled:.4f}")
print(f"Accuracy WITH scaling:    {acc_scaled:.4f}")
print(f"Improvement:              +{(acc_scaled - acc_unscaled)*100:.2f}%")
print("="*55)
print()
print("Classification Report (Scaled Model):")
print(classification_report(y_test, y_pred_scaled, target_names=['Malignant', 'Benign']))

## 9. Demo B: Finding the Optimal K via Cross-Validation

We will evaluate KNN performance for $K = 1$ to $K = 50$ using 10-fold cross-validation on the scaled Breast Cancer dataset and plot the results to identify the optimal $K$.

In [ ]:
# Evaluate K from 1 to 50
k_range = range(1, 51)
cv_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=10, scoring='accuracy')
    cv_scores.append(scores.mean())

# Find optimal K
optimal_k = k_range[np.argmax(cv_scores)]
optimal_score = max(cv_scores)

# Plot
plt.figure(figsize=(12, 5))
plt.plot(k_range, cv_scores, color='#9b59b6', linewidth=2.5, marker='o', markersize=4)
plt.axvline(x=optimal_k, color='#e74c3c', linestyle='--', linewidth=2, label=f'Optimal K = {optimal_k}')
plt.scatter([optimal_k], [optimal_score], color='#e74c3c', s=200, zorder=5, edgecolor='black')
plt.title('KNN: Cross-Validation Accuracy vs. K', fontsize=14, fontweight='bold')
plt.xlabel('Number of Neighbors (K)', fontsize=12)
plt.ylabel('10-Fold CV Accuracy', fontsize=12)
plt.legend(fontsize=12, frameon=True, facecolor='white')
plt.tight_layout()
plt.show()

print(f"Optimal K: {optimal_k}")
print(f"Best CV Accuracy: {optimal_score:.4f}")
print(f"Heuristic K (sqrt(n)): {int(np.sqrt(len(X_train_scaled)))}")

## 10. Demo C: Decision Surfaces — Overfitting vs. Underfitting

To visualize how $K$ affects the decision boundary, we will generate a simple 2D dataset and plot the decision surfaces for $K = 1$ (overfitting), optimal $K$, and $K = n$ (underfitting).

In [ ]:
# Generate 2D synthetic dataset
X_2d, y_2d = make_classification(
    n_samples=200, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.2, random_state=42
)

# Scale
scaler_2d = StandardScaler()
X_2d_scaled = scaler_2d.fit_transform(X_2d)

# Helper function for decision surface
def plot_knn_decision_surface(X, y, k, ax, title):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X, y)
    
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))
    
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[y == 0][:, 0], X[y == 0][:, 1], color='#3498db', edgecolor='k', s=50, label='Class 0')
    ax.scatter(X[y == 1][:, 0], X[y == 1][:, 1], color='#e74c3c', edgecolor='k', s=50, label='Class 1')
    
    train_acc = knn.score(X, y)
    ax.set_title(f'{title}\nTrain Acc: {train_acc:.4f}', fontsize=12, fontweight='bold')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

# Plot for K=1 (Overfit), K=optimal, K=n (Underfit)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_knn_decision_surface(X_2d_scaled, y_2d, k=1, ax=axes[0], title='K = 1 (Overfitting)')
plot_knn_decision_surface(X_2d_scaled, y_2d, k=11, ax=axes[1], title='K = 11 (Balanced)')
plot_knn_decision_surface(X_2d_scaled, y_2d, k=199, ax=axes[2], title='K = 199 (Underfitting)')

plt.tight_layout()
plt.show()

## 11. Demo D: Confusion Matrix Heatmap with Optimal K

Let's train the final KNN model using the optimal $K$ found earlier and evaluate it with a confusion matrix heatmap.

In [ ]:
# Train final model with optimal K on scaled Breast Cancer data
knn_final = KNeighborsClassifier(n_neighbors=optimal_k)
knn_final.fit(X_train_scaled, y_train)
y_pred_final = knn_final.predict(X_test_scaled)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_final)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Purples',
    xticklabels=['Predicted Malignant', 'Predicted Benign'],
    yticklabels=['Actual Malignant', 'Actual Benign']
)
plt.title(f'Confusion Matrix: KNN (K={optimal_k}) on Breast Cancer', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Actual Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

# Final report
print(f"Final KNN Model (K={optimal_k}):")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_final):.4f}")
print()
print(classification_report(y_test, y_pred_final, target_names=['Malignant', 'Benign']))

## 12. Weighted KNN

A useful enhancement to standard KNN is **distance-weighted voting**. Instead of each neighbor getting one equal vote, closer neighbors get a **higher weight** (typically the inverse of their distance):

$$w_i = \frac{1}{d(X_{\text{query}}, X_i)}$$

This means:
- A neighbor that is extremely close to the query point has a very **large** vote.
- A neighbor that is far away has a **small** vote.

In Scikit-Learn, this is configured via `weights='distance'` instead of the default `weights='uniform'`.

### Benefits of Weighted KNN
- **Reduces sensitivity to outliers:** A distant outlier neighbor has negligible voting power.
- **Smoother decision boundaries:** The transition between class regions becomes more gradual.
- **Better performance on imbalanced data:** Minority class samples that are very close to the query point can outweigh many distant majority class samples.

In [ ]:
# Compare Uniform vs Distance-Weighted KNN
knn_uniform = KNeighborsClassifier(n_neighbors=optimal_k, weights='uniform')
knn_distance = KNeighborsClassifier(n_neighbors=optimal_k, weights='distance')

knn_uniform.fit(X_train_scaled, y_train)
knn_distance.fit(X_train_scaled, y_train)

acc_uniform = accuracy_score(y_test, knn_uniform.predict(X_test_scaled))
acc_distance = accuracy_score(y_test, knn_distance.predict(X_test_scaled))

print(f"KNN (K={optimal_k}) - Uniform Weights Accuracy:  {acc_uniform:.4f}")
print(f"KNN (K={optimal_k}) - Distance Weights Accuracy: {acc_distance:.4f}")

## Summary Checklist for K-Nearest Neighbors

1. **Always Scale Features:** KNN is distance-based; use `StandardScaler` before training to ensure all features contribute equally.
2. **Select K Carefully:** Use cross-validation to find the optimal $K$. Start with $K = \sqrt{n}$ as a heuristic.
3. **Use Odd K for Binary Tasks:** Avoids ties in majority voting.
4. **Beware the Bias-Variance Trade-off:** Small $K$ overfits (noisy, jagged boundaries); large $K$ underfits (overly smooth boundaries).
5. **Consider Weighted KNN:** Set `weights='distance'` to reduce outlier sensitivity and improve performance on imbalanced data.
6. **Avoid High Dimensions:** KNN suffers from the curse of dimensionality. Use dimensionality reduction (PCA) if features exceed $\sim 20$.
7. **Not Suitable for Large Datasets:** Prediction time is $O(n \cdot d)$ per query. For big data, consider approximate nearest neighbor algorithms (e.g., KD-Trees, Ball Trees) or switch to parametric models.
8. **KNN Provides No Interpretability:** It cannot tell you *which* features drove a prediction. If feature importance is needed, use tree-based or linear models instead.